In [3]:
!pip install -q tiktoken datasets

In [4]:
import os
os.makedirs('model', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

In [5]:
%%writefile model/__init__.py


Writing model/__init__.py


In [ ]:
%%writefile model/gpt.py
"""
model/gpt.py

A small, from-scratch GPT-style decoder-only transformer.

This is Phase 1 of the project: a plain, correct, FP32 baseline implementation.
Later phases plug into this file without changing its core math:
  - Phase 2 (mixed precision):        no changes needed here -- handled by autocast in train.py
  - Phase 3 (grad checkpointing):     no changes needed here -- handled by torch.utils.checkpoint in train.py
  - Phase 4 (custom fused kernel):    model/attention.py will swap out CausalSelfAttention's
                                       internals; this file just needs to keep calling it.
  - Phase 5 (KV-cache):               generate() below is the NAIVE (no-cache) version;
                                       generate.py will add the cached version separately.

Architecture is the standard GPT-2-style decoder-only transformer:
    tokens -> embedding (+ positional embedding)
           -> N x [LayerNorm -> Self-Attention -> residual -> LayerNorm -> MLP -> residual]
           -> final LayerNorm -> linear head -> logits
"""

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 5000      
    context_length: int = 256 
    n_layers: int = 6
    n_heads: int = 6
    n_embd: int = 384           
    dropout: float = 0.1
    bias: bool = True           

    def __post_init__(self):
        assert self.n_embd % self.n_heads == 0, "n_embd must be divisible by n_heads"


class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, eps=1e-5)


class CausalSelfAttention(nn.Module):
   
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_heads

    
        self.qkv_proj = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.shape  

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

       
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,   # token i can only attend to tokens <= i
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y


class MLP(nn.Module):

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc_in = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.fc_out = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc_in(x)
        x = self.gelu(x)
        x = self.fc_out(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.context_length, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.ln_f = LayerNorm(config.n_embd, bias=config.bias)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight

        self.apply(self._init_weights)
        print(f"MiniGPT initialized: {self.num_params() / 1e6:.2f}M parameters")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):

        B, T = idx.shape
        assert T <= self.config.context_length, (
            f"sequence length {T} exceeds context_length {self.config.context_length}"
        )

        positions = torch.arange(0, T, dtype=torch.long, device=idx.device)

        tok_emb = self.token_embedding(idx)           
        pos_emb = self.position_embedding(positions)  
        x = self.dropout(tok_emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)  

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.context_length \
                else idx[:, -self.config.context_length:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # only need the last token's logits

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx



if __name__ == "__main__":
    config = GPTConfig(vocab_size=1000, context_length=64, n_layers=4, n_heads=4, n_embd=128)
    model = MiniGPT(config)

    dummy_input = torch.randint(0, config.vocab_size, (2, 32))    # batch=2, seq_len=32
    dummy_targets = torch.randint(0, config.vocab_size, (2, 32))

    logits, loss = model(dummy_input, dummy_targets)
    print(f"logits shape: {logits.shape}")   # expect (2, 32, 1000)
    print(f"loss: {loss.item():.4f}")

    loss.backward()
    print("Backward pass succeeded -- model is wired up correctly.")

    generated = model.generate(dummy_input[:, :5], max_new_tokens=10)
    print(f"generated shape: {generated.shape}")   # expect (2, 15)


Writing model/gpt.py


In [ ]:
%%writefile data/prepare_data.py

import os

import numpy as np
import tiktoken
from datasets import load_dataset

TARGET_SIZE_MB = 100        
VAL_FRACTION = 0.1           
ENCODING_NAME = "gpt2"      
OUTPUT_DIR = os.path.dirname(os.path.abspath(__file__))

TARGET_SIZE_BYTES = TARGET_SIZE_MB * 1024 * 1024


def collect_text(target_bytes):
    print(f"Streaming OpenWebText until we hit ~{target_bytes / 1024 / 1024:.0f} MB of raw text...")


    dataset = load_dataset(
        "Skylion007/openwebtext", split="train", streaming=True, trust_remote_code=True
    )

    chunks = []
    total_bytes = 0
    n_docs = 0

    for example in dataset:
        text = example["text"]
        chunks.append(text)
        total_bytes += len(text.encode("utf-8"))
        n_docs += 1

        if n_docs % 500 == 0:
            print(f"  ...{n_docs} docs, {total_bytes / 1024 / 1024:.1f} MB so far")

        if total_bytes >= target_bytes:
            break

    full_text = "\n\n".join(chunks)
    actual_mb = len(full_text.encode("utf-8")) / 1024 / 1024
    print(f"Done collecting: {n_docs} documents, {actual_mb:.1f} MB raw text")
    return full_text



def tokenize(text):
    enc = tiktoken.get_encoding(ENCODING_NAME)
    print(f"Tokenizing with '{ENCODING_NAME}' BPE (vocab_size={enc.n_vocab})...")
    ids = enc.encode_ordinary(text)
    print(f"Total tokens: {len(ids):,}")
    return ids, enc.n_vocab



def save_splits(ids, val_fraction, output_dir):
    ids = np.array(ids, dtype=np.uint16)

    n_val = int(len(ids) * val_fraction)
    train_ids = ids[:-n_val]
    val_ids = ids[-n_val:]

    train_path = os.path.join(output_dir, "train.bin")
    val_path = os.path.join(output_dir, "val.bin")

    train_ids.tofile(train_path)
    val_ids.tofile(val_path)

    print(f"train.bin: {len(train_ids):,} tokens ({os.path.getsize(train_path) / 1024 / 1024:.1f} MB)")
    print(f"val.bin:   {len(val_ids):,} tokens ({os.path.getsize(val_path) / 1024 / 1024:.1f} MB)")


# ---------------------------------------------------------------------------
if __name__ == "__main__":
    text = collect_text(TARGET_SIZE_BYTES)
    ids, vocab_size = tokenize(text)
    save_splits(ids, VAL_FRACTION, OUTPUT_DIR)

    print()
    print("Done. In model/gpt.py's GPTConfig, set:")
    print(f"    vocab_size = {vocab_size}")
    print("train.py will then memory-map train.bin / val.bin directly -- no need")
    print("to re-run this script unless you want a different data slice.")


Writing data/prepare_data.py


In [8]:
%%writefile config.py

import torch

from model.gpt import GPTConfig


model_config = GPTConfig(
    vocab_size=50257,      
    context_length=256,
    n_layers=6,
    n_heads=6,
    n_embd=384,
    dropout=0.1,
    bias=True,
)



use_amp = False                 
amp_dtype = "float16"           
use_grad_checkpointing = False  
grad_accumulation_steps = 1   


batch_size = 32               
learning_rate = 3e-4
max_iters = 2000              
warmup_iters = 100
lr_decay_iters = 2000
min_lr = 3e-5
weight_decay = 0.1
grad_clip = 1.0

eval_interval = 200           
eval_iters = 50              
log_interval = 20            


data_dir = "data"             
out_dir = "checkpoints"      
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 1337

run_name = "baseline"        


Writing config.py


In [9]:
%%writefile train.py
import csv
import math
import os
import time
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

import config
from model.gpt import MiniGPT


def get_batch(split):
    path = os.path.join(config.data_dir, f"{split}.bin")
    data = np.memmap(path, dtype=np.uint16, mode="r")

    ctx_len = config.model_config.context_length
    ix = torch.randint(len(data) - ctx_len, (config.batch_size,))
    x = torch.stack([
        torch.from_numpy(data[i:i + ctx_len].astype(np.int64)) for i in ix
    ])
    y = torch.stack([
        torch.from_numpy(data[i + 1:i + 1 + ctx_len].astype(np.int64)) for i in ix
    ])

    x, y = x.to(config.device), y.to(config.device)
    return x, y

def get_lr(it):
    if it < config.warmup_iters:
        return config.learning_rate * it / config.warmup_iters
    if it > config.lr_decay_iters:
        return config.min_lr
    decay_ratio = (it - config.warmup_iters) / (config.lr_decay_iters - config.warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config.min_lr + coeff * (config.learning_rate - config.min_lr)

def autocast_ctx():
    if not config.use_amp:
        return nullcontext()
    dtype = torch.bfloat16 if config.amp_dtype == "bfloat16" else torch.float16
    return torch.autocast(device_type=config.device, dtype=dtype)


def forward_with_optional_checkpointing(model, x, y):
    if not config.use_grad_checkpointing:
        return model(x, y)

    B, T = x.shape
    positions = torch.arange(0, T, dtype=torch.long, device=x.device)
    tok_emb = model.token_embedding(x)
    pos_emb = model.position_embedding(positions)
    h = model.dropout(tok_emb + pos_emb)

    for block in model.blocks:
        h = checkpoint(block, h, use_reentrant=False)

    h = model.ln_f(h)
    logits = model.lm_head(h)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), ignore_index=-1)
    return logits, loss


@torch.no_grad()
def estimate_loss(model):
    model.eval()
    losses = {}
    for split in ("train", "val"):
        split_losses = torch.zeros(config.eval_iters)
        for k in range(config.eval_iters):
            x, y = get_batch(split)
            with autocast_ctx():
                _, loss = model(x, y)
            split_losses[k] = loss.item()
        losses[split] = split_losses.mean().item()
    model.train()
    return losses



def log_result(results, log_path="results/logs.csv"):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    file_exists = os.path.isfile(log_path)

    with open(log_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(results.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(results)

    print(f"Logged result to {log_path}")



def train():
    torch.manual_seed(config.seed)
    os.makedirs(config.out_dir, exist_ok=True)

    model = MiniGPT(config.model_config).to(config.device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(config.use_amp and config.amp_dtype == "float16"))

    print(f"Run: {config.run_name} | device={config.device} | "
          f"amp={config.use_amp} ({config.amp_dtype if config.use_amp else 'n/a'}) | "
          f"grad_checkpointing={config.use_grad_checkpointing} | "
          f"grad_accum_steps={config.grad_accumulation_steps}")

    t0 = time.time()
    peak_mem = 0
    losses = {"train": float("nan"), "val": float("nan")}

    for it in range(config.max_iters):
        lr = get_lr(it)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        optimizer.zero_grad(set_to_none=True)
        for _ in range(config.grad_accumulation_steps):
            x, y = get_batch("train")
            with autocast_ctx():
                _, loss = forward_with_optional_checkpointing(model, x, y)
                loss = loss / config.grad_accumulation_steps
            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        if config.device == "cuda":
            peak_mem = max(peak_mem, torch.cuda.max_memory_allocated() / 1024 ** 2)

        if it % config.log_interval == 0:
            print(f"iter {it:5d} | loss {loss.item() * config.grad_accumulation_steps:.4f} | lr {lr:.2e}")

        if it % config.eval_interval == 0 or it == config.max_iters - 1:
            losses = estimate_loss(model)
            elapsed = time.time() - t0
            print(f"  eval @ iter {it}: train_loss={losses['train']:.4f} "
                  f"val_loss={losses['val']:.4f} elapsed={elapsed:.1f}s peak_mem={peak_mem:.0f}MB")

    ckpt_path = os.path.join(config.out_dir, f"{config.run_name}.pt")
    torch.save({"model": model.state_dict(), "config": config.model_config}, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

    total_time = time.time() - t0
    print(f"\nDone. Total time: {total_time:.1f}s | Peak GPU memory: {peak_mem:.0f}MB")

    results = {
        "run_name": config.run_name,
        "use_amp": config.use_amp,
        "amp_dtype": config.amp_dtype if config.use_amp else "n/a",
        "use_grad_checkpointing": config.use_grad_checkpointing,
        "grad_accumulation_steps": config.grad_accumulation_steps,
        "total_time_s": round(total_time, 1),
        "peak_mem_mb": round(peak_mem, 0),
        "final_train_loss": round(losses["train"], 4),
        "final_val_loss": round(losses["val"], 4),
    }
    log_result(results)
    return results


if __name__ == "__main__":
    train()


Writing train.py


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/mini-gpt-project'
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print('Drive mounted. Project folder:', DRIVE_DIR)

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/mini-gpt-project


In [11]:
train_bin_exists = os.path.isfile(f'{DRIVE_DIR}/data/train.bin')
val_bin_exists = os.path.isfile(f'{DRIVE_DIR}/data/val.bin')

if train_bin_exists and val_bin_exists:
    print('Found existing train.bin/val.bin in Drive — copying instead of regenerating.')
    !cp {DRIVE_DIR}/data/train.bin {DRIVE_DIR}/data/val.bin data/
else:
    print('No existing data found in Drive — generating from scratch (a few minutes)...')
    !python data/prepare_data.py
    !cp data/train.bin data/val.bin {DRIVE_DIR}/data/
    print('Saved train.bin/val.bin to Drive for next time.')

Found existing train.bin/val.bin in Drive — copying instead of regenerating.


In [13]:
import config
from train import train

config.run_name = 'baseline'
results_phase1 = train()
print(results_phase1)

MiniGPT initialized: 30.04M parameters
Run: baseline | device=cuda | amp=False (n/a) | grad_checkpointing=False | grad_accum_steps=1
iter     0 | loss 10.9100 | lr 0.00e+00
  eval @ iter 0: train_loss=10.9133 val_loss=10.9135 elapsed=15.9s peak_mem=7977MB
iter    20 | loss 10.1323 | lr 6.00e-05
iter    40 | loss 9.4032 | lr 1.20e-04
iter    60 | loss 8.2185 | lr 1.80e-04
iter    80 | loss 7.3776 | lr 2.40e-04
iter   100 | loss 7.2491 | lr 3.00e-04
iter   120 | loss 7.0193 | lr 3.00e-04
iter   140 | loss 6.8802 | lr 3.00e-04
iter   160 | loss 6.7990 | lr 2.99e-04
iter   180 | loss 6.8524 | lr 2.99e-04
iter   200 | loss 6.7055 | lr 2.98e-04
  eval @ iter 200: train_loss=6.6463 val_loss=6.6717 elapsed=128.3s peak_mem=8208MB
iter   220 | loss 6.6656 | lr 2.97e-04
iter   240 | loss 6.5661 | lr 2.96e-04
iter   260 | loss 6.5104 | lr 2.95e-04
iter   280 | loss 6.5212 | lr 2.94e-04
iter   300 | loss 6.3971 | lr 2.93e-04
iter   320 | loss 6.3634 | lr 2.91e-04
iter   340 | loss 6.3503 | lr 2.90e

In [15]:
config.use_amp = True
config.amp_dtype = 'float16'  
config.run_name = 'mixed_precision'

results_phase2 = train()
print(results_phase2)

MiniGPT initialized: 30.04M parameters
Run: mixed_precision | device=cuda | amp=True (float16) | grad_checkpointing=False | grad_accum_steps=1


/content/train.py:106: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(config.use_amp and config.amp_dtype == "float16"))


iter     0 | loss 10.9051 | lr 0.00e+00
  eval @ iter 0: train_loss=10.9133 val_loss=10.9135 elapsed=7.4s peak_mem=8208MB
iter    20 | loss 10.1317 | lr 6.00e-05
iter    40 | loss 9.4036 | lr 1.20e-04
iter    60 | loss 8.2176 | lr 1.80e-04
iter    80 | loss 7.3769 | lr 2.40e-04
iter   100 | loss 7.2492 | lr 3.00e-04
iter   120 | loss 7.0146 | lr 3.00e-04
iter   140 | loss 6.8739 | lr 3.00e-04
iter   160 | loss 6.8008 | lr 2.99e-04
iter   180 | loss 6.8498 | lr 2.99e-04
iter   200 | loss 6.6998 | lr 2.98e-04
  eval @ iter 200: train_loss=6.6455 val_loss=6.6705 elapsed=55.7s peak_mem=8208MB
iter   220 | loss 6.6666 | lr 2.97e-04
iter   240 | loss 6.5651 | lr 2.96e-04
iter   260 | loss 6.5077 | lr 2.95e-04
iter   280 | loss 6.5220 | lr 2.94e-04
iter   300 | loss 6.3965 | lr 2.93e-04
iter   320 | loss 6.3631 | lr 2.91e-04
iter   340 | loss 6.3482 | lr 2.90e-04
iter   360 | loss 6.3059 | lr 2.88e-04
iter   380 | loss 6.5139 | lr 2.86e-04
iter   400 | loss 6.3072 | lr 2.84e-04
  eval @ iter 

In [16]:
config.use_grad_checkpointing = True
config.grad_accumulation_steps = 4
config.run_name = 'memory_engineering'

results_phase3 = train()
print(results_phase3)

MiniGPT initialized: 30.04M parameters
Run: memory_engineering | device=cuda | amp=True (float16) | grad_checkpointing=True | grad_accum_steps=4
iter     0 | loss 10.8958 | lr 0.00e+00
  eval @ iter 0: train_loss=10.9132 val_loss=10.9136 elapsed=8.0s peak_mem=8208MB
iter    20 | loss 10.1427 | lr 6.00e-05
iter    40 | loss 9.2757 | lr 1.20e-04
iter    60 | loss 8.2152 | lr 1.80e-04
iter    80 | loss 7.3385 | lr 2.40e-04
iter   100 | loss 7.0042 | lr 3.00e-04
iter   120 | loss 6.9071 | lr 3.00e-04
iter   140 | loss 6.7667 | lr 3.00e-04
iter   160 | loss 6.5592 | lr 2.99e-04
iter   180 | loss 6.5539 | lr 2.99e-04
iter   200 | loss 6.3218 | lr 2.98e-04
  eval @ iter 200: train_loss=6.3910 val_loss=6.4160 elapsed=184.1s peak_mem=8208MB
iter   220 | loss 6.3873 | lr 2.97e-04
iter   240 | loss 6.2441 | lr 2.96e-04
iter   260 | loss 6.0826 | lr 2.95e-04
iter   280 | loss 6.2760 | lr 2.94e-04
iter   300 | loss 6.0479 | lr 2.93e-04
iter   320 | loss 6.0489 | lr 2.91e-04
iter   340 | loss 6.1214

In [17]:
import pandas as pd

df = pd.read_csv('results/logs.csv')
df

,run_name,use_amp,amp_dtype,use_grad_checkpointing,grad_accumulation_steps,total_time_s,peak_mem_mb,final_train_loss,final_val_loss
0,baseline,False,NaN,False,1,1170.6,8208.0,5.4243,5.5229
1,mixed_precision,True,float16,False,1,484.0,8208.0,5.4231,5.5225
2,mixed_precision,True,float16,False,1,483.7,8208.0,5.4230,5.5224
3,memory_engineering,True,float16,True,4,1763.6,8208.0,5.0081,5.0966


In [18]:
!cp -r checkpoints/* {DRIVE_DIR}/checkpoints/
!cp results/logs.csv {DRIVE_DIR}/results/logs.csv
print('Checkpoints and logs backed up to Drive.')

Checkpoints and logs backed up to Drive.
